# 06. Hyperparameter and Ablation Studies

This notebook documents controlled tuning after the main RoBERTa baselines. We avoid a blind grid because each Transformer run is expensive and because the earlier experiments already tell us which knobs are meaningful.
After the first RoBERTa results, we did not want to add advanced methods randomly. Each experiment in this notebook answers a weakness we had already observed: unstable validation loss, rare-class recall, possible overfitting, or unsafe data manipulation.


## Why These Hyperparameters Matter

- **max_length:** tokenization analysis showed short literals and recommended 32. We check 64 only as a sanity test, not as a default.
- **learning_rate:** RoBERTa fine-tuning is sensitive; 1e-5, 2e-5, and 3e-5 are the meaningful range. We kept 2e-5 as the anchor and tested 3e-5 in staged runs.
- **batch_size:** affects memory and gradient noise. We use 32 for quick checks, 64 for medium subset runs, and 128 for final full runs when GPU memory allows.
- **dropout and weight_decay:** regularization matters because validation loss rises after the best epoch.
- **warmup scheduler:** PLM fine-tuning can benefit from a gentle start before full learning rate.
- **gradient clipping:** reduces unstable updates during Transformer fine-tuning.
- **AMP:** can speed training, but we only used it in a quick check and kept it off for final reproducibility.

## Staged Search Design

**Stage A** used small subsets to check max_length, dropout, warmup, weight decay, and AMP.

**Stage B** used medium subset runs for promising learning-rate/dropout settings.

**Stage C** used one full run with the recommended controlled configuration: max_length 32, lr 2e-5, batch 128, dropout 0.1, weight decay 0.01, linear warmup ratio 0.06, and gradient clipping 1.0.

We did not try the full Cartesian product because that would multiply GPU time without a clear hypothesis.

In [ ]:
import pandas as pd
tuning = pd.read_csv('../reports/tables/v07_tuning_results.csv')
cols = ['candidate_version','stage','max_length','learning_rate','batch_size','dropout','weight_decay','warmup_ratio','scheduler','gradient_clip','use_amp','accuracy','macro_f1','weighted_f1','best_epoch']
tuning[cols]


## Final Recommended Mean-Pooling Configuration

| parameter | recommended value |
|---|---:|
| max_length | 32 |
| learning_rate | 2e-5 |
| batch_size | 128 |
| dropout | 0.1 |
| weight_decay | 0.01 |
| scheduler | linear warmup |
| warmup_ratio | 0.06 |
| gradient_clip | 1.0 |
| AMP | off |

This configuration reached validation accuracy 0.5646, macro F1 0.4942, and weighted F1 0.5491. It matched the standard mean-pooling accuracy but did not beat CLS. Therefore it is the recommended configuration for future mean-pooling experiments, but it is not the final model candidate.

![Top v07 training curves](../reports/figures/v07_roberta_mean_tuning_c_recommended_32_lr2e5_warmup006_clip_training_curves.png)


## What We Did Not Try and Why

- We did not run every combination of max_length, learning rate, batch size, dropout, weight decay, warmup ratio, scheduler, and AMP because that would be computationally expensive and poorly justified.
- We did not continue tuning max_length beyond 64 because the tokenization analysis showed that 32 already covers the literals well.
- We did not select AMP for the final run because the non-AMP run was stable and reproducible.
- We did not tune many seeds here; seed ensembling belongs to the later ensemble phase.


## Interpretation

The tuned mean-pooling run did not produce a new best model. This is still a useful negative result: it tells us that simple tuning around mean pooling is not enough to surpass CLS. The next rational direction is not a larger random grid, but error analysis, calibration, and ensembling between models that make different mistakes.

This changed how we approached the next step. Instead of assuming that one better hyperparameter would solve the task, we treated model diversity and safe data handling as more promising directions.


## Safe Data Strategies, Not Unsafe Augmentation

After tuning mean pooling, we explored whether the training data itself could be handled more safely. We intentionally did **not** use aggressive clinical text augmentation. We did not randomly delete medical words, replace terms with unverified synonyms, alter negation, or back-translate literals.

This is a responsible-AI decision. In medical literals, a small change can change the clinical meaning and therefore the ICD category. The EDA also showed that duplicate literals can have conflicting labels, so duplicate handling must be conservative rather than automatic. A class-balanced batch sampler was considered, but we kept it as future work because `WeightedRandomSampler` already tests the main sampling hypothesis with less implementation risk.


In [ ]:
import pandas as pd
v08 = pd.read_csv('../reports/tables/v08_data_strategy_results.csv')
v08


### What We Expected

We expected conservative duplicate handling to reduce repeated easy examples without hiding ambiguous cases. We expected weighted sampling to help rare categories and macro F1, but we also expected a possible drop in accuracy because the competition metric rewards every row equally and frequent categories dominate the validation set.

### What We Found

Conservative deduplication reached validation accuracy **0.5686**, slightly above standard mean pooling but still just below the CLS model. WeightedRandomSampler reached macro F1 **0.5226**, the best macro-oriented result among these safe data strategies, but its accuracy dropped to **0.5423**.

### How It Affects the Next Step

We keep conservative deduplication as a useful safe ablation and possible robustness candidate, but we do not mark it as the final model because CLS remains slightly better by validation accuracy. Weighted sampling is valuable evidence for the report: it shows the accuracy vs minority-class recall trade-off caused by imbalance.


In [ ]:
duplicates = pd.read_csv('../reports/tables/v08_roberta_mean_augmented_duplicate_report.csv')
duplicates


## Clinical Safety Conclusion

This experiment taught us that augmentation is not automatically good. For this assignment, the safest improvements are those that do not invent new clinical text: careful duplicate handling, sampling strategies, class-weighted objectives, and later ensembling or calibration. More semantic augmentation, such as synonyms or back-translation, belongs in future work unless medically validated resources or expert review are available.

This limitation leads naturally to future work: any augmentation that changes medical content would need clinical validation, not only an NLP intuition.
